# HP Quiz — RAG Pipeline
Regular (fixed-size) chunking · `all-MiniLM-L6-v2` embeddings · ChromaDB · TinyLlama-1.1B (local)

In [ ]:
import fitz  # PyMuPDF
import tiktoken
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline

EMBED_MODEL     = "all-MiniLM-L6-v2"   # ~90 MB, CPU-friendly
CHAT_MODEL      = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # ~2.2 GB
CHUNK_SIZE      = 500   # tokens
CHUNK_OVERLAP   = 50    # tokens
COLLECTION_NAME = "hp_books"
PDF_PATH        = "harrypotter.pdf"

# Load models once
embedder = SentenceTransformer(EMBED_MODEL)
generator = pipeline(
    "text-generation",
    model=CHAT_MODEL,
    dtype="auto",
    device_map="auto",
)
print("Models loaded.")

## 1. PDF extraction

In [ ]:
def extract_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    return "\n".join(page.get_text() for page in doc)

raw_text = extract_text(PDF_PATH)
print(f"Extracted {len(raw_text):,} characters")

## 2. Regular chunking

In [ ]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunks.append(enc.decode(tokens[start:end]))
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"{len(chunks):,} chunks · {CHUNK_SIZE}-token size · {CHUNK_OVERLAP}-token overlap")

## 3. Build ChromaDB index (runs once; idempotent)

In [ ]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

def build_index(chunks: list[str], collection) -> None:
    BATCH = 256
    for i in range(0, len(chunks), BATCH):
        batch = chunks[i : i + BATCH]
        embeddings = embedder.encode(batch, show_progress_bar=False).tolist()
        ids = [str(i + j) for j in range(len(batch))]
        collection.add(embeddings=embeddings, documents=batch, ids=ids)
        print(f"  indexed {min(i + BATCH, len(chunks)):,} / {len(chunks):,}", end="\r")
    print("\nDone.")

if collection.count() == 0:
    print("Building index...")
    build_index(chunks, collection)
else:
    print(f"Index already built ({collection.count():,} vectors). Skipping.")

## 4. Retrieval

In [ ]:
def retrieve(query: str, k: int = 5) -> list[str]:
    q_emb = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=k)
    return results["documents"][0]

## 5. RAG generation

In [ ]:
SYSTEM_PROMPT = "You are a Harry Potter expert. Answer concisely based only on the provided context."

def rag_query(question: str, k: int = 5) -> str:
    context = "\n\n---\n\n".join(retrieve(question, k))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ]
    prompt = generator.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    out = generator(prompt, max_new_tokens=256)
    # strip the prompt prefix, return only the generated text
    return out[0]["generated_text"][len(prompt):].strip()

## 6. Demo

In [ ]:
print(rag_query("What is the address of the Dursleys?"))

In [ ]:
print(rag_query("Generate 3 multiple-choice quiz questions about Hogwarts."))